# 📚 Unidad 1: Material Complementario - Teoría
## Módulo 02 - Manejo de Datos Faltantes y Outliers
### Laboratorio (Herramientas) - Universidad del Aconcagua

---

## 🎯 Objetivos de Aprendizaje

Al finalizar esta teoría, serás capaz de:

1. ✅ Identificar y cuantificar valores faltantes en datasets
2. ✅ Comprender los diferentes mecanismos de datos faltantes (MCAR, MAR, MNAR)
3. ✅ Aplicar estrategias de imputación apropiadas según el tipo de datos
4. ✅ Detectar outliers usando múltiples métodos estadísticos
5. ✅ Decidir cuándo eliminar, transformar o mantener outliers
6. ✅ Implementar pipelines robustos de manejo de calidad de datos

---

### 📚 Contenido

**Parte I: Datos Faltantes**
1. Tipos de valores faltantes (MCAR, MAR, MNAR)
2. Análisis de patrones de faltantes
3. Estrategias de imputación
4. Validación de imputaciones

**Parte II: Outliers**
5. Métodos de detección de outliers
6. Outliers univariados vs multivariados
7. Estrategias de tratamiento
8. Casos especiales y consideraciones de negocio

---

### ⏱️ Duración Estimada: 45 minutos

---

# 📊 PARTE I: DATOS FALTANTES

## 1️⃣ ¿Qué son los Datos Faltantes?

Los **valores faltantes** (missing values) son ausencias de información en un dataset. Pueden representarse como:
* `NaN` (Not a Number) en pandas/numpy
* `NULL` en SQL/Spark
* `None` en Python
* Celdas vacías en archivos CSV/Excel

### 🚨 Impacto de los Datos Faltantes

**Problemas que causan:**
* ❌ **Sesgo estadístico**: Medias, medianas y desviaciones incorrectas
* ❌ **Pérdida de información**: Eliminar filas puede descartar datos valiosos
* ❌ **Errores en modelos ML**: Muchos algoritmos no aceptan NaNs
* ❌ **Resultados erróneos**: Análisis basados en datos incompletos

**📊 Ejemplo con la panadería:**

| cliente_id | edad | ingreso | compras_mes | monto_total |
|------------|------|---------|-------------|-------------|
| C001 | 35 | 45000 | 8 | 12400 |
| C002 | **NaN** | 52000 | 12 | **NaN** |
| C003 | 42 | **NaN** | 5 | 8900 |
| C004 | 28 | 38000 | **NaN** | 15600 |

⚠️ **Preguntas críticas:**
* ¿La edad faltante es porque el cliente no la proporcionó o error de sistema?
* ¿El ingreso NaN significa "no aplica" o "no conocido"?
* ¿Las compras NaN son clientes nuevos o error de tracking?

---

## 2️⃣ Tipos de Mecanismos de Datos Faltantes

### 🎲 MCAR: Missing Completely At Random

**Definición**: Los valores faltantes ocurren de forma **completamente aleatoria**, sin relación con otras variables.

**Características:**
* La probabilidad de faltar es la **misma para todos** los casos
* No hay patrón sistemático
* **Es el caso más fácil de manejar**

**📊 Ejemplo:**
* Un sensor de temperatura falla aleatoriamente el 5% de las veces por defecto de hardware
* Un formulario web se cierra accidentalmente antes de completarse

**✅ Estrategia recomendada:**
* **Eliminar filas** es seguro (no introduce sesgo)
* **Imputar con media/mediana** funciona bien

---

### 🔗 MAR: Missing At Random

**Definición**: Los valores faltantes **dependen de otras variables observadas**, pero NO del valor faltante mismo.

**Características:**
* Hay un **patrón relacionado con otros datos**
* Se puede **predecir qué datos faltan** usando otras variables
* **Más común en la práctica**

**📊 Ejemplo:**
* Clientes jóvenes tienden a **no proporcionar su ingreso** en formularios
* Ventas de ciertos productos faltan más en **sucursales pequeñas**

**✅ Estrategia recomendada:**
* **Imputación condicional** (por grupos)
* **Modelos predictivos** (regresión, KNN, Random Forest)
* **MICE (Multiple Imputation by Chained Equations)**

---

### 🚫 MNAR: Missing Not At Random

**Definición**: Los valores faltantes **dependen del valor faltante mismo**. El hecho de que falte está relacionado con lo que debería ser ese valor.

**Características:**
* **El más problemático**
* Introduce **sesgo sistemático**
* No se puede corregir solo con datos observados

**📊 Ejemplos:**
* Personas con **ingresos muy bajos** tienden a no reportarlos (vergüenza)
* Pacientes con **síntomas graves** no completan encuestas de seguimiento (demasiado enfermos)
* Empleados con **bajo desempeño** no completan autoevaluaciones

**❗ Estrategia recomendada:**
* **Investigar la causa** del faltante (entrevistas, análisis de proceso)
* **Modelo de selección**: Modelar la probabilidad de que falte
* **Análisis de sensibilidad**: Probar diferentes escenarios
* Si es posible, **recolectar esos datos**

---

### 📊 Comparación Visual

| Tipo | Aleatoriedad | Predecible | Sesgo al eliminar | Dificultad |
|------|-------------|------------|-------------------|------------|
| **MCAR** | Total | No | No | 🟢 Fácil |
| **MAR** | Condicional | Sí (con otras vars) | Posible | 🟡 Media |
| **MNAR** | No aleatoria | Difícil | Sí | 🔴 Difícil |

**💡 Regla práctica:**
* Si < 5% de datos faltan → Eliminar filas suele ser seguro
* Si 5-20% faltan → Analizar patrón y decidir estrategia
* Si > 20% faltan → **Investigar causa** antes de tomar acción

---

## 3️⃣ Estrategias de Imputación

### 🧰 Método 1: Eliminación (Deletion)

#### **Listwise Deletion (eliminar fila completa)**

```python
df_clean = df.dropna()  # Elimina TODA fila con al menos un NaN
```

**✅ Cuándo usar:**
* < 5% de datos faltantes (impacto mínimo)
* MCAR confirmado
* Dataset grande (puedes perder filas)

**❌ Cuándo NO usar:**
* Muchos faltantes (> 10%)
* MAR o MNAR (introduce sesgo)
* Dataset pequeño (pérdida de información crítica)

#### **Pairwise Deletion (eliminar solo para cálculos específicos)**

```python
# Calcula correlación ignorando NaN pairwise
corr_matrix = df.corr()  # pandas ignora NaN automáticamente
```

**💡 Ventaja:** Usa máxima información disponible para cada cálculo  
**⚠️ Desventaja:** Resultados basados en diferentes subsets de datos

---

### 📊 Método 2: Imputación Estadística Simple

#### **Media/Mediana/Moda**

```python
# Imputar con media (variables numéricas)
df['edad'].fillna(df['edad'].mean(), inplace=True)

# Imputar con mediana (mejor si hay outliers)
df['ingreso'].fillna(df['ingreso'].median(), inplace=True)

# Imputar con moda (variables categóricas)
df['categoria'].fillna(df['categoria'].mode()[0], inplace=True)
```

**✅ Ventajas:**
* Rápido y simple
* No cambia la media (si imputa con media)
* Útil para datasets grandes

**❌ Desventajas:**
* **Reduce varianza** (todos los NaN tienen el mismo valor)
* **Distorsiona distribuciones**
* **Ignora relaciones** entre variables

**💡 Cuándo usar:**
* MCAR con pocos faltantes (< 5%)
* Baseline rápido
* Variables con poca importancia en el modelo

---

### 🔗 Método 3: Imputación por Grupos (Group-based)

```python
# Imputar edad con mediana por categoría de cliente
df['edad'] = df.groupby('categoria_cliente')['edad'].transform(
    lambda x: x.fillna(x.median())
)

# Imputar ingreso por combinación ciudad + profesión
df['ingreso'] = df.groupby(['ciudad', 'profesion'])['ingreso'].transform(
    lambda x: x.fillna(x.median())
)
```

**✅ Ventajas:**
* **Respeta la estructura de grupos**
* Más preciso que imputación global
* Útil para MAR

**📊 Ejemplo:**
* Edad faltante de cliente "Premium" → Imputar con mediana de clientes Premium (no todos)
* Ingreso faltante de "Mendoza + Ingeniero" → Usar mediana de ese segmento

---

### 🤖 Método 4: Imputación Predictiva (Model-based)

#### **Regresión Lineal**

```python
from sklearn.linear_model import LinearRegression

# Entrenar modelo con datos completos
df_complete = df.dropna(subset=['edad', 'ingreso', 'educacion'])
X = df_complete[['ingreso', 'educacion']]
y = df_complete['edad']

model = LinearRegression().fit(X, y)

# Predecir edades faltantes
df_missing = df[df['edad'].isna()]
df.loc[df['edad'].isna(), 'edad'] = model.predict(
    df_missing[['ingreso', 'educacion']]
)
```

#### **KNN Imputer**

```python
from sklearn.impute import KNNImputer

# Imputar usando los 5 vecinos más cercanos
imputer = KNNImputer(n_neighbors=5)
df_imputed = pd.DataFrame(
    imputer.fit_transform(df), 
    columns=df.columns
)
```

**✅ Ventajas:**
* **Preserva relaciones** entre variables
* **Más preciso** que métodos simples
* Mantiene mejor la varianza

**❌ Desventajas:**
* Más lento (requiere entrenar modelos)
* Puede overfittear si muchos NaN
* Requiere features predictivos buenos

---

### 🔄 Método 5: MICE (Multiple Imputation by Chained Equations)

```python
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

# MICE: Imputa iterativamente cada variable
imputer = IterativeImputer(max_iter=10, random_state=42)
df_mice = pd.DataFrame(
    imputer.fit_transform(df),
    columns=df.columns
)
```

**🔄 Cómo funciona:**
1. Imputa inicialmente con medias
2. Para cada variable con NaN:
   * Usa otras variables como predictores
   * Predice valores faltantes con regresión
3. Repite hasta convergencia (10 iteraciones)

**✅ Ventajas:**
* **Estado del arte** para imputación múltiple
* Preserva incertidumbre
* Funciona bien con MAR

**❌ Desventajas:**
* Lento para datasets grandes
* Requiere convergencia (puede no converger)

---

### 📊 Método 6: Forward Fill / Backward Fill (Series Temporales)

```python
# Forward fill: Usa el último valor conocido
df['temperatura'].fillna(method='ffill', inplace=True)

# Backward fill: Usa el siguiente valor conocido
df['temperatura'].fillna(method='bfill', inplace=True)

# Interpolación lineal
df['temperatura'].interpolate(method='linear', inplace=True)
```

**📊 Cuándo usar:**
* **Series temporales** con alta autocorrelación
* Sensores con mediciones frecuentes
* Variables que cambian suavemente

**⚠️ Precaución:**
* No usar si gaps largos (> 10% del periodo)
* Puede propagar errores si valor anterior ya era erróneo

---

# 🔍 PARTE II: OUTLIERS

## 4️⃣ ¿Qué son los Outliers?

Un **outlier** (valor atípico) es una observación que se **desvía significativamente** del resto de los datos.

### 📊 Visualización Conceptual

```
Distribución Normal de Precios:

       │
   300 │                                              ⭕  ← OUTLIER!
       │
   200 │
       │        ●●●●●●●●●●●●●●      
   100 │     ●●●●●●●●●●●●●●●●●●●●
       │  ●●●●●●●●●●●●●●●●●●●●●●●●
     0 │────────────────────────────
        10   20   30   40   50   60
              Precio ($)
```

### 🚨 ¿Por qué Importan los Outliers?

**Impactos negativos:**
* 📉 **Distorsionan estadísticas**: Media muy afectada (mediana es robusta)
* 🤖 **Afectan modelos ML**: Regresión lineal muy sensible
* 📊 **Ocultan patrones**: Escalas visuales dominadas por outliers
* ⚠️ **Conclusiones erróneas**: "El cliente promedio gasta $10,000" (cuando 1 gastó $100k)

**Impactos positivos:**
* 🔍 **Pueden ser los datos más interesantes**: Fraude, anomalías, oportunidades
* 💡 **Insights de negocio**: ¿Por qué algunos clientes gastan 10x más?
* ✅ **Casos edge**: Importantes para sistemas de producción
**💡 Regla de oro:** **Nunca elimines outliers sin investigar primero**

---

### 🔎 Tipos de Outliers

#### **1. Outliers Univariados**

Atípicos en **una sola variable**

**Ejemplo:**
* Edad: [25, 30, 28, 32, **150**, 29] ← 150 es outlier
* Precio: [15, 20, 18, **-5**, 22] ← -5 es erróneo

#### **2. Outliers Multivariados**

Normales individualmente, pero **combinación atípica**

**Ejemplo:**
* Edad: 25 años (normal)
* Ingreso: $150,000/mes (normal para ejecutivos)
* **Combinación**: 25 años con $150k/mes → Atípico

#### **3. Outliers Contextuales**

Normales en un contexto, atípicos en otro

**Ejemplo:**
* Venta de $50,000 en heladería:
  * En verano → Normal
  * En invierno → Outlier (investigar)

---

### 🧐 ¿Outlier o Error?

| Tipo | Causa | Acción |
|------|-------|--------|
| **Error de medición** | Sensor mal calibrado, typo | ❌ Eliminar o corregir |
| **Error de procesamiento** | Bug en código, conversión | ❌ Corregir pipeline |
| **Outlier legítimo** | Comportamiento real atípico | ✅ Mantener (o tratar especial) |
| **Evento especial** | Black Friday, pandemia | ✅ Mantener + feature "es_especial" |

**🔍 Cómo distinguir:**
1. **Validar lógica de negocio**: ¿Es posible físicamente?
   * Edad 150 años → Error
   * Venta de $1M → Investigar (quizás real)

2. **Consultar con expertos de dominio**:
   * "Vimos una transacción de $50k, ¿es posible?"
   * "¿Clientes corporativos pueden tener esos montos?"

3. **Revisar datos originales**:
   * Volver a la fuente (factura, sistema)
   * Validar con otros sistemas

---

## 5️⃣ Métodos de Detección de Outliers

### 📊 Método 1: Rango Intercuartílico (IQR)

**El método más usado en la práctica**

```python
Q1 = df['columna'].quantile(0.25)  # Percentil 25
Q3 = df['columna'].quantile(0.75)  # Percentil 75
IQR = Q3 - Q1

# Límites para outliers
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Identificar outliers
outliers = df[(df['columna'] < lower_bound) | (df['columna'] > upper_bound)]
```

**📊 Visualización:**

```
        Boxplot

    *  ← Outlier superior
    │
   ┌───────┐
   │       │  ← Q3 (percentil 75)
   │───────│  ← Mediana
   │       │  ← Q1 (percentil 25)
   └───────┘
    │
    *  ← Outlier inferior
```

**✅ Ventajas:**
* **Robusto**: No afectado por outliers extremos
* **Simple**: Fácil de entender y explicar
* **Visual**: Se mapea directo a boxplots

**❌ Limitaciones:**
* Solo univariado (una variable a la vez)
* Asume distribución simétrica
* Puede ser demasiado estricto (1.5 IQR detecta ~0.7% como outliers)

**🔧 Variantes:**
* **Outliers moderados**: 1.5 × IQR (estándar)
* **Outliers extremos**: 3.0 × IQR (más conservador)

---

### 📊 Método 2: Z-Score (Desviaciones Estándar)

**Mide cuántas desviaciones estándar se aleja un valor de la media**

```python
from scipy import stats

# Calcular Z-scores
df['z_score'] = stats.zscore(df['columna'])

# Outliers: |Z| > 3 (99.7% de datos en [-3, 3])
outliers = df[df['z_score'].abs() > 3]
```

**📊 Fórmula:**

```
Z = (x - μ) / σ

Donde:
  x  = valor observado
  μ  = media
  σ  = desviación estándar
```

**📊 Interpretación:**
* **|Z| < 2**: Normal (95% de datos)
* **2 < |Z| < 3**: Posible outlier
* **|Z| > 3**: Outlier probable

**✅ Ventajas:**
* Estandarizado (comparable entre variables)
* Interpretación probabilística clara

**❌ Limitaciones:**
* **MUY sensible a outliers** (media y σ afectadas por outliers)
* Asume distribución normal
* No funciona bien si muchos outliers

**💡 Alternativa robusta:** Usar **Median Absolute Deviation (MAD)**

```python
median = df['columna'].median()
mad = np.median(np.abs(df['columna'] - median))
modified_z = 0.6745 * (df['columna'] - median) / mad
outliers = df[modified_z.abs() > 3.5]
```

---

### 🤖 Método 3: Isolation Forest (ML)

**Algoritmo de ML para detección de anomalías**

```python
from sklearn.ensemble import IsolationForest

# Entrenar detector
iso_forest = IsolationForest(contamination=0.05, random_state=42)
df['anomaly'] = iso_forest.fit_predict(df[['columna1', 'columna2']])

# -1 = outlier, 1 = normal
outliers = df[df['anomaly'] == -1]
```

**🔍 Cómo funciona:**
1. Construye árboles de decisión aleatorios
2. Outliers se **aíslan más rápido** (menos splits)
3. Asigna score de anomalía

**✅ Ventajas:**
* **Multivariado**: Detecta outliers en combinaciones de variables
* **No asume distribución**: Funciona con cualquier tipo de datos
* **Escalable**: Eficiente para grandes datasets

**❌ Limitaciones:**
* Requiere especificar `contamination` (% esperado de outliers)
* Menos interpretable (caja negra)

---

### 🔵 Método 4: DBSCAN (Clustering)

**Outliers = puntos que no pertenecen a ningún cluster**

```python
from sklearn.cluster import DBSCAN

# Aplicar DBSCAN
dbscan = DBSCAN(eps=0.5, min_samples=5)
df['cluster'] = dbscan.fit_predict(df[['feature1', 'feature2']])

# Cluster -1 son outliers
outliers = df[df['cluster'] == -1]
```

**✅ Ventajas:**
* Multivariado
* No requiere especificar número de clusters
* Detecta outliers y clusters simultáneamente

**❌ Limitaciones:**
* Sensible a hiperparámetros (`eps`, `min_samples`)
* Lento para datasets muy grandes

---

### 📊 Método 5: Distancia de Mahalanobis (Multivariado)

**Distancia que considera correlaciones entre variables**

```python
from scipy.spatial.distance import mahalanobis

# Calcular distancia de Mahalanobis
mean = df[['col1', 'col2']].mean()
cov = df[['col1', 'col2']].cov()

df['mahal_dist'] = df.apply(
    lambda row: mahalanobis([row['col1'], row['col2']], mean, np.linalg.inv(cov)), 
    axis=1
)

# Outliers: distancia > threshold (ej. chi-cuadrado p=0.001)
threshold = chi2.ppf(0.999, df=2)  # 2 variables
outliers = df[df['mahal_dist'] > threshold]
```

**✅ Ventajas:**
* Considera **covarianza** entre variables
* Más preciso que métodos univariados

**❌ Limitaciones:**
* Asume distribución multivariada normal
* Requiere calcular matriz de covarianza (costoso)

---

### 📊 Comparación de Métodos

| Método | Tipo | Interpretabilidad | Velocidad | Mejor para |
|---------|------|------------------|-----------|------------|
| **IQR** | Univariado | 🟢 Alta | 🟢 Rápido | EDA inicial, boxplots |
| **Z-Score** | Univariado | 🟢 Alta | 🟢 Rápido | Distribución normal |
| **Isolation Forest** | Multivariado | 🟡 Media | 🟢 Rápido | Datasets grandes, ML |
| **DBSCAN** | Multivariado | 🟡 Media | 🔴 Lento | Clusters + outliers |
| **Mahalanobis** | Multivariado | 🔴 Baja | 🔴 Lento | Datos correlacionados |

**💡 Estrategia recomendada:**
1. **EDA inicial**: IQR + boxplots (rápido, visual)
2. **Confirmación**: Z-score o MAD
3. **Análisis multivariado**: Isolation Forest
4. **Validación manual**: Revisar outliers detectados

---

## 6️⃣ Estrategias de Tratamiento de Outliers

### 🛠️ Estrategia 1: Eliminación

**Cuándo eliminar:**
* ❌ **Error confirmado** (edad 150, precio negativo)
* ❌ **Menos del 1%** del dataset
* ❌ **No aportan información** (datos corruptos)

```python
# Eliminar outliers identificados
df_clean = df[~((df['columna'] < lower_bound) | (df['columna'] > upper_bound))]

print(f"Filas eliminadas: {len(df) - len(df_clean)} ({(len(df) - len(df_clean))/len(df):.1%})")
```

**⚠️ Precauciones:**
* **Documenta qué eliminaste y por qué**
* Guarda outliers en tabla separada para auditoría
* Nunca elimines silenciosamente sin notificar al equipo

---

### 🔄 Estrategia 2: Transformación

#### **Capping/Winsorizing (reemplazar con límite)**

```python
# Reemplazar outliers con percentiles
percentile_99 = df['columna'].quantile(0.99)
percentile_01 = df['columna'].quantile(0.01)

df['columna_capped'] = df['columna'].clip(lower=percentile_01, upper=percentile_99)
```

**✅ Ventajas:**
* No pierdes filas
* Reduce impacto de extremos
* Mantiene relaciones con otras variables

**📊 Cuándo usar:**
* Outliers legítimos pero extremos
* Modelos sensibles (regresión lineal)
* Cuando no puedes perder datos

#### **Transformaciones Matemáticas**

```python
# Log transform (reduce asimetría)
df['columna_log'] = np.log1p(df['columna'])  # log(1+x) para evitar log(0)

# Square root
df['columna_sqrt'] = np.sqrt(df['columna'])

# Box-Cox (automatiza mejor transformación)
from scipy.stats import boxcox
df['columna_boxcox'], lambda_param = boxcox(df['columna'] + 1)
```

**✅ Ventajas:**
* **Normaliza distribuciones** sesgadas
* Reduce impacto de outliers naturalmente
* Mejora performance de modelos

**📊 Cuándo usar:**
* Distribuciones con sesgo positivo (long tail)
* Variables de ingreso, ventas, población
* Antes de modelado ML

---

### 🏷️ Estrategia 3: Segmentación

**Tratar outliers como segmento especial**

```python
# Crear categoría "outlier"
df['segmento'] = 'normal'
df.loc[(df['monto'] > upper_bound), 'segmento'] = 'alto_valor'
df.loc[(df['monto'] < lower_bound), 'segmento'] = 'bajo_valor'

# Analizar por segmento
df.groupby('segmento').agg({
    'monto': ['count', 'mean', 'sum'],
    'cliente_id': 'nunique'
})
```

**✅ Ventajas:**
* **Insights de negocio**: ¿Quiénes son los clientes de alto valor?
* No pierdes información
* Puedes crear estrategias diferenciadas

**📊 Ejemplo:**
* Clientes que gastan > $10,000 → Segmento VIP
* Marketing diferenciado
* Análisis separado

---

### 🤖 Estrategia 4: Modelos Robustos

**Usar algoritmos que sean robustos a outliers**

| Modelo | Sensibilidad a Outliers | Alternativa Robusta |
|--------|------------------------|---------------------|
| **Regresión Lineal** | 🔴 Alta | Huber Regression, RANSAC |
| **K-Means** | 🔴 Alta | DBSCAN, Gaussian Mixture |
| **Media** | 🔴 Alta | Mediana, Media Truncada |
| **Árboles de Decisión** | 🟢 Baja | Ya son robustos |
| **Random Forest** | 🟢 Baja | Ya son robustos |

```python
# Regresión robusta (menos sensible a outliers)
from sklearn.linear_model import HuberRegressor

model_robust = HuberRegressor(epsilon=1.35)  # Tolerancia a outliers
model_robust.fit(X, y)
```

---

### 📊 Estrategia 5: Imputación (reemplazar con predicción)

```python
# Predecir outliers como si fueran missing
from sklearn.ensemble import RandomForestRegressor

# Marcar outliers como NaN
df_temp = df.copy()
df_temp.loc[df_temp['columna'] > upper_bound, 'columna'] = np.nan

# Imputar con modelo
features = ['feature1', 'feature2', 'feature3']
model = RandomForestRegressor()

# Entrenar con datos normales
df_train = df_temp[df_temp['columna'].notna()]
model.fit(df_train[features], df_train['columna'])

# Predecir outliers
df_impute = df_temp[df_temp['columna'].isna()]
df_temp.loc[df_temp['columna'].isna(), 'columna'] = model.predict(df_impute[features])
```

**💡 Cuándo usar:**
* Outliers sospechosos (probablemente errores)
* Quieres mantener el número de filas
* Tienes features predictivos buenos

---

### 📊 Workflow de Decisión

```
┌────────────────────────────┐
│   Detectar Outlier       │
└──────────┬─────────────────┘
            │
            v
┌────────────────────────────┐
│   ¿Es un Error?          │
└─────────┬───────────────────┘
         │
    Sí   │   No
    │    │
    v    v
┌──────────────────────────────┐
│ ELIMINAR o CORREGIR      │
└──────────────────────────────┘
         │
         v
    ┌──────────────────────────┐
    │  ¿Afecta Modelo ML?    │
    └──────┬────────────────────┘
         │
    Sí   │   No
    │    │
    v    v
┌──────────────────────────────────┐
│ TRANSFORMAR o MODELO      │  │
│ ROBUSTO                   │  │
└──────────────────────────────────┘  │
                               v
                          ┌─────────────────────────┐
                          │  MANTENER y SEGMENTAR   │
                          │  (análisis especial)    │
                          └─────────────────────────┘
```

---

## ✅ Resumen y Mejores Prácticas

### 📊 Datos Faltantes: Quick Reference

| Situación | Mecanismo | Estrategia Recomendada |
|-----------|-----------|------------------------|
| < 5% faltan, aleatorio | MCAR | Eliminar filas |
| 5-20% faltan, patrón | MAR | Imputación por grupos o predictiva |
| > 20% faltan, sistemático | MNAR | Investigar causa, modelo de selección |
| Series temporales | - | Forward/backward fill, interpolación |
| Categóricas | - | Moda o categoría "desconocido" |

### 📊 Outliers: Quick Reference

| Situación | Estrategia Recomendada |
|-----------|------------------------|
| Error confirmado | Eliminar o corregir |
| < 1% outliers legítimos | Eliminar (si no son importantes) |
| Outliers importantes para negocio | Mantener + segmentar |
| Afectan modelo ML | Transformar (log, capping) o modelo robusto |
| Distribución sesgada | Transformación (log, Box-Cox) |
| Multivariados | Isolation Forest o DBSCAN |

---

### 💡 Mejores Prácticas Universales

#### **SIEMPRE:**
1. ✅ **Documenta todo**: Qué eliminaste, imputaste o transformaste
2. ✅ **Visualiza antes y después**: Boxplots, histogramas, scatter plots
3. ✅ **Valida con expertos**: Consulta antes de eliminar datos
4. ✅ **Guarda versiones**: Dataset crudo, limpio, transformado
5. ✅ **Reproduce**: Código debe ser reproducible

#### **NUNCA:**
1. ❌ Elimines outliers sin investigar
2. ❌ Imputes sin entender el patrón de faltantes
3. ❌ Uses media si hay outliers (usa mediana)
4. ❌ Ignores el contexto de negocio
5. ❌ Sobrescribas datos originales sin backup

---

### 🔄 Workflow Completo Recomendado

```python
# 1. EDA inicial
print("=" * 60)
print("REPORTE DE CALIDAD DE DATOS")
print("=" * 60)

# Valores faltantes
print("\n1. VALORES FALTANTES:")
missing = df.isnull().sum()
print(missing[missing > 0].sort_values(ascending=False))

# Outliers (IQR)
print("\n2. OUTLIERS (Método IQR):")
for col in df.select_dtypes(include=[np.number]).columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    
    if len(outliers) > 0:
        print(f"  {col}: {len(outliers)} outliers ({len(outliers)/len(df):.1%})")

# 2. Visualizaciones
import matplotlib.pyplot as plt
import seaborn as sns

# Matriz de valores faltantes
sns.heatmap(df.isnull(), cbar=False)
plt.title("Patrón de Valores Faltantes")
plt.show()

# Boxplots para outliers
df.boxplot(figsize=(12, 6))
plt.xticks(rotation=45)
plt.title("Detección de Outliers")
plt.show()

# 3. Decisiones de limpieza
# (Basadas en análisis anterior)

# 4. Aplicar transformaciones
# (Imputación, eliminación, transformaciones)

# 5. Validar resultados
print("\n3. RESULTADO DE LIMPIEZA:")
print(f"  Filas antes: {len(df)}")
print(f"  Filas después: {len(df_clean)}")
print(f"  Filas eliminadas: {len(df) - len(df_clean)}")
print(f"  Valores faltantes restantes: {df_clean.isnull().sum().sum()}")
```

---

### 🚀 Próximos Pasos

**En el curso:**
* ✅ Aplica estos métodos en la **práctica** del Módulo 02
* ✅ Continúa con **Módulo 03: Análisis de Correlaciones**
* ✅ Integra con EDA Avanzado del Módulo 01

**Recursos adicionales:**
* 📖 [Pandas Missing Data](https://pandas.pydata.org/docs/user_guide/missing_data.html)
* 📖 [Scikit-learn Imputation](https://scikit-learn.org/stable/modules/impute.html)
* 📖 [PyOD (Python Outlier Detection)](https://github.com/yzhao062/pyod)

---

**Universidad del Aconcagua - Facultad de Ciencias Económicas y Jurídicas**  
**Licenciatura en Analítica de Negocios**  
**Mendoza, Argentina 🇦🇷**